# DASH-Q perplexity comparison

Quantise the smallest available Gemma (`google/gemma-3-270m`) with DASH-Q and compare
wikitext-2 perplexity against four baselines: **FP16**, **GGUF RTN k-quants**
(via the bundled fork of llama.cpp), **AWQ** (`autoawq`), and **GPTQ** (`gptqmodel`).

Why this model: the user asked for "the smallest possible network like Gemma 3n".
Gemma 3n is multi-billion params with matformer + per-layer embeddings and does
not export cleanly to plain GGUF. `gemma-3-270m` is the smallest text-only
Gemma 3 variant, has a plain decoder, and round-trips through every backend
below.

## Paper correctness review

Before benchmarking, cross-check `src/dashq/dash_q.py` against the paper
(arXiv:2604.13806v1, Kim et al., EuroMLSys '26). Algorithm 1 and Appendix A
give the closed-form solutions.

| Paper element | Code | Verdict |
|---|---|---|
| Diagonal Hessian $h_{jj}=\sum_k x_{kj}^2$ (Sec. 5, Eq. 8) | `dash_q.py:145` `(X**2).sum(dim=0)` | match |
| Initialisation Eq. 12: $s^{(0)}=(\max W-\min W)/(2^b-1)$, $z^{(0)}=-\min W$ | `dash_q.py:64-70` | match |
| Integer refinement Eq. 1: $Q=\mathrm{clip}(\lfloor(W+z)/s\rceil,0,2^b-1)$ | `dash_q.py:80, 103` | match |
| Eq. 10: $s^*=\mathrm{Cov}_h(W,Q)/(\mathrm{Var}_h(Q)+\lambda)$ | `dash_q.py:92` | match |
| Eq. 11: $z^*=s^*\bar q_h-\bar w_h$ | `dash_q.py:96` | match |
| Section 6.1 hyperparams: $T=9$, $\alpha=0.5$, $\lambda=10^{-2}$ | `dash_q.py:29-31` | match |
| Group sizes: 128 / 64 / 32 for 4 / 3 / 2-bit | `dash_q.py:22` | match |
| Smoothing $\alpha$ (Sec. 6.1 "$\alpha=0.5$") | `dash_q.py:99-100` blends $s,z$ across iterates | match (paper omits the blend from the Alg. 1 pseudocode but names $\alpha=0.5$ in Sec. 6.1) |

Conclusion: the algorithm is implemented faithfully. The GGUF packers in
`src/dashq/export*.py` are storage formats, not algorithm; outside this
review's scope.

## 1. Setup

In [ ]:
import sys
import gc
import math
import time
import pathlib
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO = pathlib.Path('..').resolve()
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(REPO))

from dashq.dash_q import quantize_layer, dequantize, PAPER_GROUP_SIZES

MODEL_ID = 'google/gemma-3-270m'
BITS = [4, 3, 2]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
CTX = 2048
STRIDE = 1024
N_CALIB = 16
SEQ_LEN = 2048
OUT_DIR = REPO / 'notebooks' / 'ppl_artifacts'
OUT_DIR.mkdir(exist_ok=True)

# Pre-quantised baselines on HuggingFace Hub. Set to a repo id to skip the
# local quantise() call and pull the artefacts. Leave as None to quantise
# from scratch (needs GPU + autoawq/gptqmodel).
# Example overrides for a model that does have published quants:
#   PRECOMPILED_AWQ = 'TheBloke/Llama-2-7B-AWQ'
#   PRECOMPILED_GPTQ = {4: 'TheBloke/Llama-2-7B-GPTQ', 3: None, 2: None}
PRECOMPILED_AWQ: str | None = None
PRECOMPILED_GPTQ: dict[int, str | None] = {4: None, 3: None, 2: None}
print(f'device={DEVICE}, dtype={DTYPE}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
wiki_test = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
wiki_text = '\n\n'.join(wiki_test['text'])
eval_ids = tokenizer(wiki_text, return_tensors='pt').input_ids
print(f'eval tokens: {eval_ids.shape[1]:,}')

## 2. Shared evaluator

Sliding-window NLL over wikitext-2 test, fed to whichever HF-style model is
passed in. Every method below is evaluated with the same function so the
numbers are directly comparable.

In [ ]:
@torch.no_grad()
def eval_ppl(model, input_ids, ctx=CTX, stride=STRIDE) -> float:
    model.eval()
    nlls = []
    n_tokens = 0
    prev_end = 0
    for begin in range(0, input_ids.size(1), stride):
        end = min(begin + ctx, input_ids.size(1))
        trg_len = end - prev_end
        ids = input_ids[:, begin:end].to(model.device)
        target_ids = ids.clone()
        target_ids[:, :-trg_len] = -100
        out = model(ids, labels=target_ids)
        nll = out.loss.float() * trg_len
        nlls.append(nll)
        n_tokens += trg_len
        prev_end = end
        if end == input_ids.size(1):
            break
    return float(torch.exp(torch.stack(nlls).sum() / n_tokens))

## 3. Calibration tokens

Same 16x2048 wikitext-2 train batch used by every PTQ method . a fair
comparison.

In [ ]:
wiki_train = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
calib_text = '\n\n'.join(t for t in wiki_train['text'] if t.strip())
calib_ids = tokenizer(calib_text, return_tensors='pt').input_ids[0]
calib_batches = calib_ids[: N_CALIB * SEQ_LEN].view(N_CALIB, SEQ_LEN)
print(f'calib: {calib_batches.shape}')

## 4. FP16 baseline

In [ ]:
results = []

def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
t0 = time.time()
fp16_ppl = eval_ppl(model, eval_ids)
results.append({'method': 'FP16', 'bits': 16, 'ppl': fp16_ppl, 'time_s': time.time() - t0})
print(f'FP16 PPL = {fp16_ppl:.4f}')
del model; free()

## 5. DASH-Q (in-place HF eval)

We hook every `nn.Linear` outside `lm_head`, capture the same calibration
activations the other methods see, then run `dashq.quantize_layer` per
weight and replace the weight with `dequantize(Q, S, Z)`. The dequantised
values are stored back in float16 . they encode exactly what the GGUF
pack-then-unpack pipeline would produce, minus the GGUF rounding.

In [ ]:
import torch.nn as nn

def capture_activations(model, batches):
    acts = {}
    hooks = []
    def mk_hook(name):
        def hook(_m, inputs, _out):
            x = inputs[0].detach().reshape(-1, inputs[0].shape[-1]).float().cpu()
            if name in acts:
                acts[name] = torch.cat([acts[name], x], dim=0)
            else:
                acts[name] = x
        return hook
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear) and 'lm_head' not in name:
            hooks.append(mod.register_forward_hook(mk_hook(name)))
    with torch.no_grad():
        for i in range(batches.size(0)):
            model(batches[i:i+1].to(model.device))
    for h in hooks:
        h.remove()
    return acts

def dashq_quantize_inplace(model, acts, bits):
    gs = PAPER_GROUP_SIZES[bits]
    for name, mod in model.named_modules():
        if not isinstance(mod, nn.Linear) or 'lm_head' in name:
            continue
        if name not in acts:
            continue
        W = mod.weight.data.float().cpu()
        if W.shape[1] % gs != 0:
            continue  # skip rows that don't divide cleanly
        X = acts[name]
        Q, S, Z = quantize_layer(W, X, b=bits, group_size=gs)
        W_hat = dequantize(Q, S, Z, gs)
        mod.weight.data.copy_(W_hat.to(mod.weight.dtype).to(mod.weight.device))
    return model

In [ ]:
for b in BITS:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
    acts = capture_activations(model, calib_batches)
    t0 = time.time()
    model = dashq_quantize_inplace(model, acts, b)
    q_time = time.time() - t0
    ppl = eval_ppl(model, eval_ids)
    results.append({'method': 'DASH-Q', 'bits': b, 'ppl': ppl, 'time_s': q_time})
    print(f'DASH-Q {b}-bit  PPL={ppl:.4f}  (quant {q_time:.1f}s)')
    del model, acts; free()

## 6. RTN (round-to-nearest) baseline

Per-group min-max affine, no calibration. Same group sizes DASH-Q uses, so
the only thing being measured is the DASH-Q solver's added value.

In [ ]:
def rtn_quantize_inplace(model, bits):
    gs = PAPER_GROUP_SIZES[bits]
    qmax = (1 << bits) - 1
    for name, mod in model.named_modules():
        if not isinstance(mod, nn.Linear) or 'lm_head' in name:
            continue
        W = mod.weight.data.float()
        if W.shape[1] % gs != 0:
            continue
        Wg = W.view(W.shape[0], -1, gs)
        wmin = Wg.min(dim=-1, keepdim=True).values
        wmax = Wg.max(dim=-1, keepdim=True).values
        s = ((wmax - wmin) / qmax).clamp(min=1e-8)
        z = -wmin
        Q = ((Wg + z) / s).round().clamp(0, qmax)
        W_hat = (s * Q - z).view_as(W)
        mod.weight.data.copy_(W_hat.to(mod.weight.dtype))
    return model

for b in BITS:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
    t0 = time.time()
    rtn_quantize_inplace(model, b)
    q_time = time.time() - t0
    ppl = eval_ppl(model, eval_ids)
    results.append({'method': 'RTN', 'bits': b, 'ppl': ppl, 'time_s': q_time})
    print(f'RTN {b}-bit  PPL={ppl:.4f}  (quant {q_time:.2f}s)')
    del model; free()

## 7. AWQ baseline

Two modes:

- **local**: `autoawq` quantises on the fly . needs GPU + ~1-2 min for a tiny model
- **precompiled**: set `PRECOMPILED_AWQ` in the setup cell to a HuggingFace
  repo id (e.g. `TheBloke/...-AWQ`) and we just download + evaluate

`autoawq` public kernels are INT4 only, so this row is always 4-bit.

In [ ]:
try:
    from awq import AutoAWQForCausalLM
    have_awq = True
except ImportError:
    have_awq = False
    print('autoawq not installed - skipping AWQ. run: uv pip install autoawq')

awq_src = None
if PRECOMPILED_AWQ:
    awq_src = PRECOMPILED_AWQ
    print(f'AWQ: using precompiled {PRECOMPILED_AWQ}')
elif have_awq and DEVICE == 'cuda':
    awq_dir = OUT_DIR / 'awq-4bit'
    if not awq_dir.exists():
        model = AutoAWQForCausalLM.from_pretrained(MODEL_ID, safetensors=True)
        model.quantize(tokenizer, quant_config={
            'zero_point': True, 'q_group_size': 128, 'w_bit': 4, 'version': 'GEMM'})
        model.save_quantized(str(awq_dir))
        tokenizer.save_pretrained(str(awq_dir))
        del model; free()
    awq_src = str(awq_dir)

if awq_src and have_awq:
    t0 = time.time()
    model = AutoAWQForCausalLM.from_quantized(awq_src, fuse_layers=False).model
    ppl = eval_ppl(model, eval_ids)
    results.append({'method': 'AWQ', 'bits': 4, 'ppl': ppl, 'time_s': time.time() - t0})
    print(f'AWQ 4-bit  PPL={ppl:.4f}')
    del model; free()
else:
    print('skipping AWQ (no precompiled id and no local autoawq/GPU)')

## 8. GPTQ baseline

Same two modes per bit width. `PRECOMPILED_GPTQ[bits]` to skip the local
search, otherwise `gptqmodel` quantises on the fly. `gptqmodel` supports
4/3/2-bit.

In [ ]:
try:
    from gptqmodel import GPTQModel, QuantizeConfig
    have_gptq = True
except ImportError:
    have_gptq = False
    print('gptqmodel not installed - skipping GPTQ. run: uv pip install gptqmodel')

for b in BITS:
    pre = PRECOMPILED_GPTQ.get(b)
    src = None
    if pre:
        src = pre
        print(f'GPTQ {b}-bit: using precompiled {pre}')
    elif have_gptq and DEVICE == 'cuda':
        gptq_dir = OUT_DIR / f'gptq-{b}bit'
        gs = PAPER_GROUP_SIZES[b]
        if not gptq_dir.exists():
            cfg = QuantizeConfig(bits=b, group_size=gs)
            m = GPTQModel.load(MODEL_ID, cfg)
            calib_ds = [tokenizer.decode(calib_batches[i].tolist()) for i in range(N_CALIB)]
            m.quantize(calib_ds, batch_size=1)
            m.save(str(gptq_dir))
            del m; free()
        src = str(gptq_dir)
    if src and have_gptq:
        t0 = time.time()
        m = GPTQModel.load(src).model
        ppl = eval_ppl(m, eval_ids)
        results.append({'method': 'GPTQ', 'bits': b, 'ppl': ppl, 'time_s': time.time() - t0})
        print(f'GPTQ {b}-bit  PPL={ppl:.4f}')
        del m; free()
    else:
        print(f'skipping GPTQ {b}-bit (no precompiled id and no local gptqmodel/GPU)')

## 9. GGUF k-quants (RTN via llama.cpp)

Uses the bundled `llama.cpp/build/bin/llama-quantize` against an F16 GGUF.
Evaluated through `llama-cpp-python` so it goes through the same `eval_ppl`
function (we wrap the `Llama` object so it exposes `.device` and a
transformers-like `__call__(input_ids, labels=...)`).

In [ ]:
import subprocess
from huggingface_hub import snapshot_download
LLAMA_DIR = REPO / 'llama.cpp'
LLAMA_BIN = LLAMA_DIR / 'build' / 'bin'
CONVERT_PY = LLAMA_DIR / 'convert_hf_to_gguf.py'
RTN_TYPE = {2: 'Q2_K', 3: 'Q3_K', 4: 'Q4_1'}

have_llamacpp_bin = (LLAMA_BIN / 'llama-quantize').exists()
print('llama-quantize present:', have_llamacpp_bin)

if have_llamacpp_bin:
    f16_gguf = OUT_DIR / 'f16.gguf'
    if not f16_gguf.exists():
        local = pathlib.Path(snapshot_download(MODEL_ID, allow_patterns=['*.json','*.model','*.safetensors']))
        subprocess.run(['uv','run','python',str(CONVERT_PY),str(local),
                        '--outfile',str(f16_gguf),'--outtype','f16'], check=True)
    gguf_paths = {}
    for b in BITS:
        out = OUT_DIR / f'rtn-{b}bit.gguf'
        if not out.exists():
            subprocess.run([str(LLAMA_BIN/'llama-quantize'), str(f16_gguf), str(out), RTN_TYPE[b]], check=True)
        gguf_paths[b] = out

In [ ]:
try:
    from llama_cpp import Llama
    have_llcpp = True
except ImportError:
    have_llcpp = False
    print('llama-cpp-python not installed - skipping GGUF PPL. run: uv pip install llama-cpp-python')

@torch.no_grad()
def gguf_ppl(gguf_path, input_ids, ctx=CTX, stride=STRIDE):
    llm = Llama(model_path=str(gguf_path), n_ctx=ctx, logits_all=True, verbose=False)
    ids = input_ids[0].tolist()
    nll_sum, n = 0.0, 0
    prev = 0
    for begin in range(0, len(ids), stride):
        end = min(begin + ctx, len(ids))
        chunk = ids[begin:end]
        trg = end - prev
        llm.reset()
        llm.eval(chunk)
        logits = np.asarray(llm._scores)  # (len, vocab)
        # next-token prediction: logits[i] predicts chunk[i+1]
        log_probs = logits - np.log(np.exp(logits - logits.max(-1, keepdims=True)).sum(-1, keepdims=True)) - logits.max(-1, keepdims=True)
        targets = np.asarray(chunk[1:])
        tok_ll = log_probs[np.arange(len(targets)), targets]
        keep = tok_ll[-trg:] if trg <= len(tok_ll) else tok_ll
        nll_sum += -keep.sum()
        n += len(keep)
        prev = end
        if end == len(ids):
            break
    del llm; free()
    return math.exp(nll_sum / n)

if have_llcpp and have_llamacpp_bin:
    for b in BITS:
        t0 = time.time()
        ppl = gguf_ppl(gguf_paths[b], eval_ids)
        results.append({'method': f'GGUF {RTN_TYPE[b]}', 'bits': b, 'ppl': ppl, 'time_s': time.time()-t0})
        print(f'GGUF {RTN_TYPE[b]}  PPL={ppl:.4f}')

## 10. Results

In [ ]:
df = pd.DataFrame(results)
df['delta_vs_fp16'] = df['ppl'] - fp16_ppl
df = df.sort_values(['bits', 'method'], ascending=[False, True]).reset_index(drop=True)
df.to_csv(OUT_DIR / 'results.csv', index=False)
df

In [ ]:
plot_df = df[df['method'] != 'FP16'].copy()
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(plot_df, x='bits', y='ppl', hue='method', ax=ax)
ax.axhline(fp16_ppl, ls='--', c='black', label=f'FP16 = {fp16_ppl:.2f}')
ax.set_title(f'{MODEL_ID} wikitext-2 perplexity by method and bit width')
ax.set_ylabel('Perplexity (lower is better)')
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / 'ppl_chart.png', dpi=120)
plt.show()

## Sanity expectations

- FP16 PPL on wikitext-2 for `gemma-3-270m` lands around 25-40 . if it's far
  outside that range the evaluator is broken.
- 4-bit DASH-Q should sit within a few percent of FP16.
- 2-bit is where DASH-Q is supposed to win: RTN typically blows up to >10^2,
  DASH-Q should stay much closer to FP16. Paper Table 1 shows this pattern
  consistently across larger models.
- DASH-Q vs GPTQ at 2-bit is the headline comparison from Section 6.2.